## Environmental programming using Python

## Assignment topic: Suitability Mapping of Nature-Based Solutions Locations to Tackle Hydroclimatic Extremes and Water Quality Degradation Using Machine Learning 

Group 4: Elias Zgheib, Ndra Malky, Rashmi Krishnamurthy, Teju Kumar Nagaraju

This notebook utilizes the previously extracted pixel values (From Task-2A) and map the pixel values of Managed Aquifer Recharge (MAR) points. It also adds the suitability column to the dataframe, existing MAR pixels is given 1 and non MAR pixels is given as 0. If more than one MAR points are located within the pixel, this scripts is now defined to select only one MAR point in the pixel


# Task 2B — Map MAR Points to Pixels + Create Binary Dataset

## 1) Library imports

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio

## 2) User inputs (Edit this cell) 
Update your input/output paths here.

In [ ]:
# Existing pixel table (from Task 2A output)
PIXEL_CSV = r"E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv"

# Managed Aquifer Recharge (MAR) shapefile
MAR_SHP   = r"E:\VUB\Final\Boundaries\MAR_EU.shp"
MAR_FIELD = "main_mar_t"  # attribute field containing MAR type/category

# Reference raster for point to pixel_id conversion (must match the pixel_id logic used in Task 2A)
REF_RASTER = r"E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif"

# Outputs
OUT_MAR_SAMPLES = r"E:\VUB\Final\PixelDataFrames\mar_samples_pixels.csv"
OUT_BINARY_CSV  = r"E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv"

# If MAR locations are static across all years
STATIC_MAR = True
MAR_YEAR = 2014   # used only if STATIC_MAR = False

print("PIXEL_CSV:", PIXEL_CSV)
print("MAR_SHP:", MAR_SHP)
print("REF_RASTER:", REF_RASTER)
print("OUT_MAR_SAMPLES:", OUT_MAR_SAMPLES)
print("OUT_BINARY_CSV:", OUT_BINARY_CSV)
print("STATIC_MAR:", STATIC_MAR, "| MAR_YEAR:", MAR_YEAR)


## 3) Pre checks
Ensures files exist and output folders are writable. Generative AI tool was used to generate this part of code

In [ ]:
def ensure_dir(path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

missing = False
for p, name in [(PIXEL_CSV,"PIXEL_CSV"), (MAR_SHP,"MAR_SHP"), (REF_RASTER,"REF_RASTER")]:
    if not os.path.exists(p):
        print("Missing:", name, "->", p)
        missing = True
    else:
        print("Found:", name)

ensure_dir(OUT_MAR_SAMPLES)
ensure_dir(OUT_BINARY_CSV)
print("Output dirs ready")

if missing:
    raise FileNotFoundError("Fix missing paths in Config cell and rerun.")


## 4) Helpers (category normalization + stable mode)
Generative AI tool was used to generate this part of code

In [ ]:
def norm(s):
    """Normalize category string (case-insensitive, trim spaces)."""
    return " ".join(str(s).strip().lower().split())

def mode_category(s: pd.Series):
    """Stable mode (alphabetical tie-break)."""
    vc = s.value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

print("Helpers loaded.")


## 5) Load inputs + quick preview

In [ ]:
df_pixels = pd.read_csv(PIXEL_CSV)
gmar = gpd.read_file(MAR_SHP)

print("Pixel table rows:", len(df_pixels))
print("Pixel table cols:", list(df_pixels.columns)[:20], "..." if len(df_pixels.columns) > 20 else "")
print("MAR points:", len(gmar))
print("MAR columns:", list(gmar.columns))

display(df_pixels.head(5))
display(gmar.head(5))


## 6) Step A — Sample MAR shapefile → pixels (creates `OUT_MAR_SAMPLES`)
Generative AI tool was used to generate this part of code

In [ ]:
print("\n=== STEP A: MAR SAMPLING ===")

if MAR_FIELD not in gmar.columns:
    raise ValueError(f"Field '{MAR_FIELD}' not found in MAR shapefile")

valid_pixels = set(df_pixels["pixel_id"].unique())
print("Unique pixels in PIXEL_CSV:", len(valid_pixels))

# ---- Normalize MAR categories ----
CATEGORIES_STD = [
    "In-Channel Modification",
    "Induced Bank Filtration",
    "Rainwater and Run-off Harvesting",
    "Spreading Methods",
    "Well, Shaft and Borehole Recharge",
]

norm_to_std = {norm(c): c for c in CATEGORIES_STD}
cat_to_code = {c: i for i, c in enumerate(CATEGORIES_STD)}

gmar2 = gmar.dropna(subset=[MAR_FIELD]).copy()
gmar2["_cat_norm"] = gmar2[MAR_FIELD].astype(str).apply(norm)
gmar2 = gmar2[gmar2["_cat_norm"].isin(norm_to_std)].copy()
gmar2["MAR_label"] = gmar2["_cat_norm"].map(norm_to_std)
gmar2.drop(columns="_cat_norm", inplace=True)

print("Total MAR points:", len(gmar))
print("After category filtering:", len(gmar2))

# ---- Convert MAR points to pixel_id ----
with rasterio.open(REF_RASTER) as ref:
    if gmar2.crs != ref.crs:
        gmar2 = gmar2.to_crs(ref.crs)

    xs = gmar2.geometry.x.to_numpy()
    ys = gmar2.geometry.y.to_numpy()
    rows, cols = rasterio.transform.rowcol(ref.transform, xs, ys)
    W = ref.width

gmar2["row"] = rows
gmar2["col"] = cols
gmar2["pixel_id"] = gmar2["row"].astype("int64") * W + gmar2["col"].astype("int64")

# ---- Guard: keep only pixels inside the pixel table domain ----
before = len(gmar2)
gmar2 = gmar2[gmar2["pixel_id"].isin(valid_pixels)].copy()
print(f"After pixel guard: {len(gmar2)} (removed {before - len(gmar2)})")

# ---- One MAR label per pixel (mode) ----
gmar_mode = (
    gmar2.groupby("pixel_id")["MAR_label"]
        .apply(mode_category)
        .reset_index()
)
gmar_mode["MAR_code"] = gmar_mode["MAR_label"].map(cat_to_code).astype(int)

# ---- Merge with pixel table ----
if STATIC_MAR:
    samples = df_pixels.merge(gmar_mode, on="pixel_id", how="inner")
else:
    df_year = df_pixels[df_pixels["year"] == MAR_YEAR].copy()
    samples = df_year.merge(gmar_mode, on="pixel_id", how="inner")

samples["pixel_year_id"] = samples["pixel_id"].astype(str) + "_" + samples["year"].astype(str)

samples.to_csv(OUT_MAR_SAMPLES, index=False)

print("Unique MAR pixels:", len(gmar_mode))
print("MAR samples saved:", OUT_MAR_SAMPLES)
display(samples.head(10))


## 7) Step B — Create binary dataset for ML (creates `OUT_BINARY_CSV`)
Generative AI tool was used to generate this part of code

In [ ]:
print("\n=== STEP B: BINARY DATASET ===")

df_all = pd.read_csv(PIXEL_CSV)
df_mar = pd.read_csv(OUT_MAR_SAMPLES)

print("Total pixel records:", len(df_all))
print("Total MAR sample records:", len(df_mar))

# Default: unsuitable
df_all["MAR_suitable"] = 0
mar_pixels = set(df_mar["pixel_id"].unique())
df_all.loc[df_all["pixel_id"].isin(mar_pixels), "MAR_suitable"] = 1

# Auto-detect feature columns
EXCLUDE = {
    "year", "pixel_id", "row", "col", "lon", "lat",
    "MAR_label", "MAR_code", "pixel_year_id", "MAR_suitable"
}
feature_cols = [c for c in df_all.columns if c not in EXCLUDE]

final_cols = ["year", "pixel_id", "lon", "lat"] + feature_cols + ["MAR_suitable"]
df_final = df_all[final_cols].dropna(subset=feature_cols, how="any")

df_final.to_csv(OUT_BINARY_CSV, index=False)

print("Binary dataset saved:", OUT_BINARY_CSV)
print("Class distribution:")
print(df_final["MAR_suitable"].value_counts())
display(df_final.head(10))


## 8) Optional: quick checks (duplicates + missing)
Generative AI tool was used to generate this part of code

In [ ]:
# Check if any pixel_id appears multiple times in mar samples (should be 1 per pixel_id)
if os.path.exists(OUT_MAR_SAMPLES):
    s = pd.read_csv(OUT_MAR_SAMPLES)
    dup = s["pixel_id"].duplicated().sum()
    print("Duplicate pixel_id in mar_samples_pixels.csv:", dup)

# Missingness summary in final dataset
if os.path.exists(OUT_BINARY_CSV):
    d = pd.read_csv(OUT_BINARY_CSV, nrows=500000)  # sample
    miss = d.isna().mean().sort_values(ascending=False).head(15)
    print("\nTop missingness (sample):")
    print(miss)
